# 评价决策类
## 1. 层次分析法(AHP)
用于确定权重
***
抛砖引玉 - 选择策略:
- 分别按不同指标归一化, 消除不同指标数量级的影响
- 权重(如何科学设计权重? - 层次分析法)

层次分析法: 对一些较为复杂、较为模糊的问题作出决策的简易方法, 它特别适用于那些**难于完全定量分析**的问题. 运用层次分析法建模，大体上可按下面四个步骤进行：
- 建立递阶层次结构模型
- 构造出判断矩阵(两两比较). 关于主对角线互为倒数(正互反矩阵).(但由于每次都是两两比较, 可能出现矛盾, 所以必须进行一致性检验)
- **一致性检验**. 一致矩阵: 满足矩阵各行（列）成倍数关系的正互反矩阵. 一致性检验原理：检验我们构造的判断矩阵和一致矩阵是否有太大差别. 如果**差别可以容忍**, 则通过一致性检验
- 求权重后进行评价

### 构造判断矩阵与一致性检验

In [13]:
import numpy as np

# 定义矩阵A
# A = np.array([[1, 2, 3, 5], [1/2, 1, 1/2, 2], [1/3, 2, 1, 1/2], [1/5, 1/2, 2, 1]])
A = np.array([[1, 2, 3, 5], [1/2, 1, 1/2, 2], [1/3, 2, 1, 2], [1/5, 1/2, 1/2, 1]])

n = A.shape[0]  # 获取A的行
# n == 4

# 求出最大特征值以及对应的特征向量
eig_val, eig_vec = np.linalg.eig(A)  # eig_val是特征值， eig_vec是特征向量
Max_eig = max(abs(eig_val))  # 求特征值的最大值. 如果有虚数, 则比较的是模长.
# 注意numpy中的max比较的是实部大小, 和matlab不同. 需要abs. 不过实对称矩阵的特征值一定是实数, 因此也无所谓

CI = (Max_eig - n) / (n - 1)
RI = [0, 0, 0.52, 0.89, 1.12, 1.26, 1.36, 1.41, 1.46, 1.49, 1.52, 1.54, 1.56, 1.58, 1.59]  
# 注意哦，这里的RI最多支持 n = 15

CR = 0 if n == 1 or n == 2 else CI / RI[n - 1]

print('一致性指标CI =', CI)
print('一致性比例CR =', CR)

if CR < 0.10:
    print('因为CR < 0.10, 所以该判断矩阵A的一致性可以接受!')
else:
    print('注意CR >= 0.10, 因此该判断矩阵A需要进行修改!')


一致性指标CI = 0.03761001273071566
一致性比例CR = 0.04225844127046703
因为CR < 0.10, 所以该判断矩阵A的一致性可以接受!


### 算数平均法求权重

In [14]:
import numpy as np

# 定义判断矩阵A
A = np.array([[1, 2, 3, 5], [1/2, 1, 1/2, 2], [1/3, 2, 1, 2], [1/5, 1/2, 1/2, 1]])

# 计算每列的和
ASum = np.sum(A, axis=0)

# 获取A的行和列, 这里A是一个方阵
n, _ = A.shape

# 归一化. 在Python中二维数组除以一维数组, 会自动将一维数组扩展为二维数组相同的形状, 然后进行逐元素的除法运算
Stand_A = A / ASum

# 各列相加到同一行
ASumr = np.sum(Stand_A, axis=1)

# 计算权重向量
weights = ASumr / n # 实际上也是归一化. 这里的和刚好是n

print(weights)


[0.48885991 0.18192996 0.2318927  0.09731744]


### 几何平均法求权重

In [15]:
import numpy as np

# 定义判断矩阵A
A = np.array([[1, 2, 3, 5], [1/2, 1, 1/2, 2], [1/3, 2, 1, 2], [1/5, 1/2, 1/2, 1]])

# 获取A的行和列
n, _ = A.shape

# 将A中每一行元素相乘得到一列向量
prod_A = np.prod(A, axis=1)

# 将新的向量的每个分量开n次方等价求1/n次方
prod_n_A = np.power(prod_A, 1/n)

# 归一化处理
re_prod_A = prod_n_A / np.sum(prod_n_A)

# 展示权重结果
print(re_prod_A)


[0.49492567 0.17782883 0.22724501 0.1000005 ]


### 特征值法求权重

In [16]:
import numpy as np

# 定义判断矩阵A
A = np.array([[1, 2, 3, 5], [1/2, 1, 1/2, 2], [1/3, 2, 1, 2], [1/5, 1/2, 1/2, 1]])

# 获取A的行和列
n, _ = A.shape

# 求出特征值和特征向量
eig_values, eig_vectors = np.linalg.eig(A)

# 找出最大特征值的索引
max_index = np.argmax(eig_values)

# 找出对应的特征向量
max_vector = eig_vectors[:, max_index]

# 对特征向量进行归一化处理,得到权重
weights = max_vector / np.sum(max_vector)

# 输出权重
print(weights)

[0.4933895 +0.j 0.17884562+0.j 0.230339  +0.j 0.09742588+0.j]


## 2. Topsis
用于多指标决策: A, B, C三位候选明星, 该选择哪个当选封面
***
**层次分析法的决策层不能太多, 而且构造判断矩阵相对主观.**

因此引入: Topsis法. 通过归一化后(去量纲化)的数据规范化矩阵, 找出多个目标中最优目标和最劣目标(分别用理想解和反理想解表示), 分别计算各评价目标与理想解和反理想解的距离, 获得各目标与理想解的贴近度, 按理想解贴近度的大小排序, 以此作为评价目标优劣的依据. 贴近度取值在0～1之间, 该值愈接近1, 表示相应的评价目标越接近最优水平; 反之, 该值愈接近0, 表示评价目标越接近最劣水平. 步骤:
- 将原始矩阵正向化(使所有指标都是越大越好, 即转化为极大型指标)*(详见ppt)*
- 正向矩阵标准化. 标准化的方法有很多种, 其主要目的就是去除量纲的影响, 保证不同评价指标在同一数量级, 且数据大小排序不变
- 标准化后, 还需给不同指标加上权重, 采用的权重确定方法有层次分析法、熵权法、Delphi法、对数最小二乘法等
- 计算得分并归一化.

### 原始矩阵正向化

In [ ]:
import numpy as np
INDEX_TYPE = ['INDEX_MAX', 'INDEX_MIN', 'INDEX_MID', 'INDEX_RANGE'] # 四种指标的种类:
# 极大型指标
# 极小型指标
# 中间型指标, 越接近某个值越好
# 区间型指标, 落在某个区间最好
BEST = 165 # 适用于第三个指标
RANGE_LOW = 90 # 适用于第四个指标
RANGE_HIGH = 100

CANDICATE_INFO_MATRIX = np.array([ # 列数: 指标数
    [9, 10, 175, 120],
    [8, 7, 164, 80],
    [6, 3, 157, 90]
], dtype=np.float64) # dtype转换很关键, 否则numpy自动指定, 若是int容易出现10/20=0
CANDIDATE_NUM, INDEX_NUM = CANDICATE_INFO_MATRIX.shape

# 正向化
X = np.zeros(shape=(CANDIDATE_NUM, 1))
for num, index_type in enumerate(INDEX_TYPE, start=0): 
    col = np.array(CANDICATE_INFO_MATRIX[:, num]) # col仍是行向量
    if index_type == 'INDEX_MAX':  # 如果当前指标为极大型，则直接使用原值
        v = col
    elif index_type == 'INDEX_MIN':  # 如果当前指标为极小型
        v = max(col) - col # 数 - 行向量, 自动扩展
    elif index_type == 'INDEX_MID':  # 如果当前指标为中间型
        abs_diff_matrix = abs(col - BEST)
        M = max(abs_diff_matrix)
        if M == 0: M = 1
        v = 1 - abs_diff_matrix/M
    elif index_type == 'INDEX_RANGE':  # 如果当前指标为区间型
        M = max(RANGE_LOW - min(col), max(col) - RANGE_HIGH)
        if M == 0: M = 1
        v = col.copy()
        v[col < RANGE_LOW] = 1 - (RANGE_LOW - v[col < RANGE_LOW]) / M
        v[(RANGE_LOW <= col) & (col <= RANGE_HIGH)] = 1 # 布尔索引用&表示and, |表示or
        v[col > RANGE_HIGH] = 1 - (v[col > RANGE_HIGH] - RANGE_HIGH) / M

    if num == 0:
        X = v.reshape(-1, 1)  # 如果是第一个指标，直接替换X数组. -1代表让numpy自己推测长度
    else:
        X = np.hstack([X, v.reshape(-1, 1)])  # 如果不是第一个指标，则将新指标列拼接到X数组上

X


array([[9. , 0. , 0. , 0. ],
       [8. , 3. , 0.9, 0.5],
       [6. , 7. , 0.2, 1. ]])

### 正向矩阵标准化

In [18]:
# L2标准化
X = X / np.sqrt(np.sum(np.square(X), axis=0))  # 对每一列数据进行归一化处理，即除以该列的欧几里得范数

X

array([[0.66896473, 0.        , 0.        , 0.        ],
       [0.59463532, 0.3939193 , 0.97618706, 0.4472136 ],
       [0.44597649, 0.91914503, 0.21693046, 0.89442719]])

### 计算得分归一化

In [19]:
# 此时如果有权重, 直接把权重乘到X对应的列. 这里假设权重相同, 省略这一步
# 计算得分
x_max = np.max(X, axis=0)  # 计算标准化矩阵每列的最大值
x_min = np.min(X, axis=0)  # 计算标准化矩阵每列的最小值
d_plus = np.sqrt(np.sum(np.square(X - x_max), axis=1)) # 一定要注意axis=1得到的也是行向量
d_minus = np.sqrt(np.sum(np.square(X - x_min), axis=1)) #  如果后续要进行加减运算需要reshape
score = d_minus / (d_plus + d_minus)
score_nomalized = 100 * score / np.sum(score)

score_nomalized

array([ 8.88636674, 45.65334106, 45.46029221])

## 3. 熵权法
用于确定权重
***
前面提到的层次分析法是一种主观确定权重的方法, 而熵权法是一种客观确定权重的方法, 可以靠数据本身得出权重. 基本逻辑: 指标的变异程度越小, 所反映的信息量也越少, 其对应的权值也应该越低. 即: 如果某项指标的值全部相等, 则该指标在综合评价中不起作用. 步骤:

- 数据正向化
- 数据标准化(指标值是否存在负数会影响标准化方法. 详见ppt)
- 计算第j项指标下第i个样本所占的比重
- 计算熵权

### 正向数据标准化

In [20]:
import numpy as np

# 假设完成了正向化, 得到正向矩阵X
X = np.array([
    [9. , 0. , 0. , 0. ],
    [8. , 3. , 0.9, 0.5],
    [6. , 7. , 0.2, 1. ]
    ], dtype=np.float64) # 注: 不同候选人按行排列, 不同指标按列排列

CANDIDATE_NUM, INDEX_NUM = X.shape

# 对矩阵X进行标准化处理，得到标准化矩阵Z
Z = (X - np.min(X, axis=0)) / (np.max(X, axis=0) - np.min(X, axis=0)) if np.any(X < 0) \
    else X / np.sqrt(np.sum(np.square(X), axis=0))

Z

array([[0.66896473, 0.        , 0.        , 0.        ],
       [0.59463532, 0.3939193 , 0.97618706, 0.4472136 ],
       [0.44597649, 0.91914503, 0.21693046, 0.89442719]])

### 计算每个样本所占比重

In [21]:
Z = Z / np.sum(Z, axis=0) # 其实这样看, Z为正项矩阵时上一步的标准化没有意义
Z

array([[0.39130435, 0.        , 0.        , 0.        ],
       [0.34782609, 0.3       , 0.81818182, 0.33333333],
       [0.26086957, 0.7       , 0.18181818, 0.66666667]])

### 计算熵权

In [22]:
Z[Z != 0] *= np.log(Z[Z != 0]) # 利用布尔索引只ln非0部分, 为0的部分不变
e = -1/np.log(CANDIDATE_NUM) * np.sum(Z, axis=0)
d = 1 - e # 效用值
w = d / np.sum(d)
w

array([0.00856537, 0.30716152, 0.39326471, 0.2910084 ])

## 4. 模糊综合评价
用于多指标决策, 比TOPSIS主观
***
如果我们问一个人的性别、身高、体重，可能很容易的得到答案, 性别一般而言非男即女, 身高和体重是可以精确测量的, 这些是确定性概念; 但是如果问到大与小, 长与短, 美与丑等概念, 就不好确定了, 多大算大? 多小算小? 这些就是模糊性概念.

为在数学上衡量这些模糊概念, 引入模糊集合A. 𝜇_A是A的隶属函数, x是A中的元素, 𝜇_A(x)是x对模糊集A的隶属度. 简单来说, 隶属度就是元素属于某个模糊集合的程度, 而隶属函数就是用来确定隶属度的函数. 𝜇_A(x)=0.5最模糊.

和TOPSIS一样, 模糊集也分偏小(年轻), 中间(中年)和偏大型(年老).

**隶属函数的确定方法**: 1)模糊统计法 2)借助已有的客观尺度(如恩格尔系数) 3)指派法(凭主观意愿)

### 对于指标较少的, 可以使用一级模糊综合评价

In [23]:
import numpy as np

R = np.array([ # 从四个方面考核员工为优良中差烂
    [0.1, 0.5, 0.4, 0, 0],
    [0.2, 0.5, 0.2, 0.1, 0],
    [0.2, 0.5, 0.3, 0, 0],
    [0.2, 0.6, 0.2, 0, 0],
    ]) # 按行为因素集, 按列为评语集(可归一化). 填入隶属度

# 权重分配为
A = np.array([0.25, 0.2, 0.25, 0.3]) # 对应各因素的权重
# 评价结果
# np.dot是Numpy库中的一个函数，用于计算两个数组的点积。对于一维数组，它计算的是这两个数组的内积
# 对于二维数组（矩阵），它计算的是矩阵乘法
B = np.dot(A, R)
B # 综合评价结果为良好

array([0.175, 0.53 , 0.275, 0.02 , 0.   ])

### 在指标非常多时, 可采用多层次模糊综合评价, 低级的B作为高级的R的一行

In [24]:
# 影响运行费用的各因素的单因素评价矩阵为:
R23 = np.array([
    [0.18, 0.14, 0.18, 0.14, 0.13, 0.23],
    [0.15, 0.20, 0.15, 0.25, 0.10, 0.15],
    [0.25, 0.12, 0.13, 0.12, 0.18, 0.20],
    [0.16, 0.15, 0.21, 0.11, 0.20, 0.17],
    [0.23, 0.18, 0.17, 0.16, 0.15, 0.11],
    [0.19, 0.13, 0.12, 0.12, 0.11, 0.33],
    [0.17, 0.16, 0.15, 0.08, 0.25, 0.19]])

# 权重分配为
A23 = np.array([0.20, 0.15, 0.10, 0.10, 0.20, 0.15, 0.10]) # 对应各因素集的权重
# 评价结果
B23 = np.dot(A23, R23) # 一级模糊综合评价

# 产品情况的二级评判如下：
R1 = np.array([
    [0.12, 0.18, 0.17, 0.23, 0.13, 0.17],
    [0.15, 0.13, 0.18, 0.25, 0.12, 0.17],
    [0.14, 0.13, 0.16, 0.18, 0.20, 0.19],
    [0.12, 0.14, 0.15, 0.17, 0.19, 0.23],
    [0.16, 0.12, 0.13, 0.25, 0.18, 0.16]])
A1 = np.array([0.15, 0.40, 0.25, 0.10, 0.10])
B1 = np.dot(A1, R1)
# 销售能力二级评判如下：
R2 = np.array([
    [0.13, 0.15, 0.14, 0.18, 0.16, 0.25],
    [0.12, 0.16, 0.13, 0.17, 0.19, 0.23],
    B23,
    [0.14, 0.13, 0.15, 0.16, 0.18, 0.24],
    [0.16, 0.15, 0.15, 0.17, 0.18, 0.19]])
A2 = np.array([0.2, 0.15, 0.25, 0.25, 0.15])
B2 = np.dot(A2, R2)

# 市场需求的二级评判
R3 = np.array([
    [0.15, 0.14, 0.13, 0.18, 0.14, 0.26],
    [0.16, 0.15, 0.18, 0.14, 0.16, 0.21]])
A3 = np.array([0.55, 0.45])
B3 = np.dot(A3, R3)

# 3、三级模糊综合评判
R = np.array([B1, B2, B3])
A = np.array([0.4, 0.3, 0.3])
B = np.dot(A, R)

B

array([0.147975 , 0.1427875, 0.1561625, 0.1862875, 0.1575375, 0.20985  ])

## 5. 灰色关联分析(GRA)

1)用于判断指标之间的关联度: 工业产值和农业产值, 哪个对GDP的影响更大?

2)用于确定权重.
***
灰色系统理论是1982年由邓聚龙创立的一门边缘性学科(interdisciplinary). 灰色系统用颜色深浅反映信息量的多少。说一个系统是黑色的，就是说这个系统信息量太少；说一个系统是白色的，就是说这个系统是清楚的，信息量充足。处于黑白之间的系统，或说**信息不完全**的系统，称为灰色系统, 简称灰系统。

所谓关联分析，就是系统地分析因素。回答的问题是：**某个包含多种因素的系统中，哪些因素是主要的，哪些是次要的；哪些因素影响大，哪些因素影响小；哪些因素是明显的，哪些因素是潜在的；哪些是需要发展的，哪些需要抑制**......

灰色关联度分析(Grey Relation Analysis，GRA)，是一种多因素统计分析的方法, 基本思想是根据**序列曲线几何形状的相似程度**来判断其联系是否紧密。灰色关联分析方法弥补了采用数理统计方法作系统分析所导致的缺憾, 它对样本量的多少和样本有无规律都同样适用, 而且计算量小, 十分方便, 更不会出现量化结果与定性分析结果不符的情况。

### 灰色关联分析

In [ ]:
import numpy as np

# 按行为时间等, 按列为不同指标
A = np.array([[55, 24, 10], 
              [65, 38, 22], 
              [75, 40, 18], 
              [100, 50, 20]],
              dtype=np.float64)

# 预处理: 除以列均值
A_norm = A / np.mean(A, axis=0)

# 母序列: 例如实例1)中的GDP. 分析与"谁"的关联
Y = A_norm[:, 0] # 得到的也是行向量

# 子序列: "谁"与母序列的关联
X = A_norm[:, 1:]

absX0_Xi = np.abs(X - Y.reshape(-1, 1))

# 计算两级最小差a
a = np.min(absX0_Xi)

# 计算两级最大差b
b = np.max(absX0_Xi)

# 分辨系数取0.5
rho = 0.5

# 计算子序列中各个指标与母序列的关联系数
ksi = (a + rho * b) / (absX0_Xi + rho * b)

np.mean(ksi, axis=0)

array([0.76966578, 0.60058464])